# Tahap 1 — Eksplorasi Data (EDA) Dataset RSUA

**File ini bertujuan untuk:**
1. Memastikan jumlah citra RSUA *Validated* sesuai ketentuan (32 Normal, 53 Pneumonia, dan 207 COVID-19 dikeluarkan).
2. Memverifikasi pasangan citra dan *mask* dokter radiolog lengkap 1-ke-1 dengan ukuran 256 x 256 piksel.
3. Menampilkan contoh visual garis batas paru buatan dokter radiolog.

**Daftar Isi:**
- [1. Persiapan Lingkungan](#1-persiapan-lingkungan)
- [2. Verifikasi Pasangan Citra & Mask Dokter](#2-verifikasi-pasangan-citra-mask-dokter)
- [3. Visualisasi Contoh Batas Paru Radiolog](#3-visualisasi-contoh-batas-paru-radiolog)

<a id="1-persiapan-lingkungan"></a>
## 1. Persiapan Lingkungan

Mengecek keberadaan folder dataset RSUA *Validated* dan menyiapkan folder penyimpanan hasil EDA.
- `load_config()` & `ensure_output_dirs()`: memuat jalur folder dataset RSUA dan folder keluaran.

In [ ]:
# Sel 1: Setup lingkungan & path RSUA Validated
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive  # type: ignore
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")

for candidate in [Path.cwd(), Path.cwd().parent, Path("/content/drive/MyDrive/LAB_TA_GradCAM_Pneumonia/code")]:
    if (candidate / "src" / "paths.py").exists():
        sys.path.insert(0, str(candidate))
        break

from src.paths import get_project_paths, ensure_output_dirs, load_config

CFG = load_config()
PATHS = ensure_output_dirs(cfg=CFG)
PROJECT_ROOT = PATHS["project_root"]
RSUA_VAL_ROOT = PATHS["rsua_validated"]
FIG_DIR = PATHS["outputs"] / "figures" / "eda"
TAB_DIR = PATHS["outputs"] / "tables" / "eda"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT  : {PROJECT_ROOT}")
print(f"RSUA Val Root : {RSUA_VAL_ROOT}")
print(f"Eksis?        : {RSUA_VAL_ROOT.exists()}")

<a id="2-verifikasi-pasangan-citra-mask-dokter"></a>
## 2. Verifikasi Pasangan Citra & Mask Dokter

Memeriksa jumlah berkas, kecocokan pasangan gambar-*mask*, ukuran 256 x 256 piksel, dan nilai biner (0 atau 255) pada seluruh kelas RSUA.
- `Image.open()` & `np.unique()`: memeriksa dimensi citra dan memastikan *mask* hanya berisi nilai 0 dan 255.

In [ ]:
# Sel 2: Verifikasi Pasangan Citra-Mask, Resolusi 256x256, dan Sifat Biner
classes_info = [
    ("Non_Covid", "Normal", 32),
    ("Non_Covid_Pneumonia", "Pneumonia", 53),
    ("Covid", "Covid (Excluded)", 207),
]

rsua_records = []

for folder_name, role_label, target_n in classes_info:
    f_dir = RSUA_VAL_ROOT / folder_name
    if not f_dir.exists():
        print(f"[PERINGATAN] Folder {folder_name} tidak ditemukan di {f_dir}")
        continue

    img_dir = f_dir / "images"
    mask_dir = f_dir / "masks"

    img_files = sorted([p for p in img_dir.glob("*.bmp") if not p.name.startswith(".")])
    mask_files = sorted([p for p in mask_dir.glob("*.bmp") if not p.name.startswith(".")])

    # Verifikasi ukuran dan biner pada seluruh citra di kelas evaluasi (Normal & Pneumonia)
    all_256 = True
    all_binary = True
    for im_p, mk_p in zip(img_files, mask_files):
        with Image.open(im_p) as im:
            if im.size != (256, 256):
                all_256 = False
        with Image.open(mk_p) as mk:
            if mk.size != (256, 256):
                all_256 = False
            u_vals = np.unique(np.array(mk))
            if not set(u_vals.tolist()).issubset({0, 255}):
                all_binary = False

    rsua_records.append({
        "folder": folder_name,
        "eval_role": role_label,
        "target_count": target_n,
        "actual_images": len(img_files),
        "actual_masks": len(mask_files),
        "pairs_matched": len(img_files) == len(mask_files),
        "all_256x256": all_256,
        "mask_binary_0_255": all_binary,
        "status_qc": "PASS" if (len(img_files) == target_n and len(img_files) == len(mask_files) and all_256 and all_binary) else "FAIL",
    })

rsua_df = pd.DataFrame(rsua_records)
out_tab = TAB_DIR / "rsua_validated_summary.csv"
rsua_df.to_csv(out_tab, index=False)

print("=== HASIL VERIFIKASI RSUA VALIDATED ===")
print(rsua_df.to_string(index=False))
print(f"Tabel tersimpan di: {out_tab}")

<a id="3-visualisasi-contoh-batas-paru-radiolog"></a>
## 3. Visualisasi Contoh Batas Paru Radiolog

Menampilkan contoh citra asli, *mask* biner, dan garis batas paru dokter pada sampel Normal dan Pneumonia.
- `ax.contour()`: menggambar garis kontur batas paru di atas citra asli dan menyimpannya ke `rsua_sample_contour_overlay.png`.

In [ ]:
# Sel 3: Visualisasi Overlay Sampel Citra CXR RSUA dan Ground Truth Mask
fig, axes = plt.subplots(2, 3, figsize=(10, 7), dpi=150)

sample_specs = [
    ("Non_Covid", "Normal", 0),
    ("Non_Covid_Pneumonia", "Pneumonia", 1),
]

for folder_name, role_name, row_idx in sample_specs:
    f_dir = RSUA_VAL_ROOT / folder_name
    if not f_dir.exists():
        continue
    img_files = sorted([p for p in (f_dir / "images").glob("*.bmp") if not p.name.startswith(".")])
    mask_files = sorted([p for p in (f_dir / "masks").glob("*.bmp") if not p.name.startswith(".")])
    if not img_files:
        continue

    im = Image.open(img_files[0]).convert("L")
    mk = Image.open(mask_files[0]).convert("L")
    im_arr = np.array(im, dtype=np.float32) / 255.0
    mk_arr = np.array(mk, dtype=np.uint8)

    # Citra Asli
    axes[row_idx, 0].imshow(im_arr, cmap="gray")
    axes[row_idx, 0].set_title(f"{role_name}: CXR Asli ({img_files[0].name})", fontsize=9)
    axes[row_idx, 0].axis("off")

    # Mask Biner
    axes[row_idx, 1].imshow(mk_arr, cmap="gray")
    axes[row_idx, 1].set_title(f"{role_name}: Ground Truth Mask", fontsize=9)
    axes[row_idx, 1].axis("off")

    # Overlay Kontur Paru
    axes[row_idx, 2].imshow(im_arr, cmap="gray")
    axes[row_idx, 2].contour(mk_arr > 127, levels=[0.5], colors=["#00FF66"], linewidths=1.2)
    axes[row_idx, 2].set_title(f"{role_name}: Overlay Kontur Radiolog", fontsize=9)
    axes[row_idx, 2].axis("off")

fig.tight_layout()
fig_out = FIG_DIR / "rsua_sample_contour_overlay.png"
fig.savefig(fig_out)
plt.close(fig)
print(f"Visualisasi overlay tersimpan di: {fig_out}")